In [0]:
# Definir la ruta de la capa Landing
landing_path = "abfss://landing@adlsolist.dfs.core.windows.net/olist_raw/"

# Lista de archivos a explorar
files = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

# Exploración básica y validación de lectura
for file in files:
    print(f"--- Explorando: {file} ---")
    file_path = f"{landing_path}{file}"
    
    # Leer el CSV crudo
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(file_path)
    
    # Mostrar el esquema inferido y el volumen real
    df.printSchema()
    print(f"Total de registros: {df.count()}\n")
    
    # Mostrar una muestra de 5 registros (usar display para la vista de tabla en Databricks)
    display(df.limit(5))

In [0]:
from pyspark.sql.functions import col, sum as _sum

print("Iniciando Profiling de Calidad de Datos...\n")

for file in files:
    file_path = f"{landing_path}{file}"
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)
    
    total_rows = df.count()
    distinct_rows = df.distinct().count()
    duplicados = total_rows - distinct_rows
    
    print(f"--- Tabla: {file} ---")
    print(f"Total registros: {total_rows} | Duplicados exactos: {duplicados}")
    
    # Calcular nulos por columna
    nulos_df = df.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
    nulos_row = nulos_df.collect()[0]
    
    hay_nulos = False
    for c in df.columns:
        nulos = nulos_row[c]
        if nulos > 0:
            porcentaje = (nulos / total_rows) * 100
            print(f"  - {c}: {nulos} nulos ({porcentaje:.2f}%)")
            hay_nulos = True
            
    if not hay_nulos:
        print("  - Sin valores nulos en ninguna columna.")
    print("-" * 40 + "\n")